# Robustness — residual-amplitude sweep

Public reproducibility version for the manuscript *Mapping-Class-Group Learning for Diffeomorphisms*.

**Important:** the revised manuscript reports candidate selection by unpenalized validation loss (equivalently, `lambda = 0`). The original exploratory notebooks also logged a reference word-length penalty of `1e-7`; that legacy setting did not change the selected candidate in the reported runs.

Run Jupyter from the repository root so that relative output paths under `results/` resolve as intended.



# Step 5: Residual-amplitude sweep

This experiment examines how mapping-class selection changes as the amplitude of the identity-component residual deformation increases.

The target is
\[
f_\varepsilon = F_\varepsilon\circ AB,
\]
where
\[
F_\varepsilon(x_1,x_2)
=
\left(
x_1+\varepsilon\sin\bigl(2\pi(x_1-x_2)\bigr),
\;
x_2+\varepsilon\sin(2\pi x_2)
\right)
\pmod 1.
\]

The Jacobian determinant is
\[
\det DF_\varepsilon
=
\left(1+2\pi\varepsilon\cos(2\pi(x_1-x_2))\right)
\left(1+2\pi\varepsilon\cos(2\pi x_2)\right).
\]

Thus \(F_\varepsilon\) remains in the natural diffeomorphic regime as long as
\[
\varepsilon<\frac{1}{2\pi}\approx0.15915.
\]

We therefore sweep
\[
\varepsilon\in\{0,\;0.05,\;0.10,\;0.14,\;0.155,\;0.158\},
\]
approaching the diffeomorphism boundary without crossing it.

To isolate the effect of residual amplitude, all other settings are held fixed:
- hidden mapping-class element: \(G_\star=AB\);
- original 9-candidate set;
- uniform sampling on \(T^2\);
- \(2000/1000/1000\) train/validation/test samples;
- PNN \(4\to128\to128\to2\);
- \(\tanh\) activations;
- residual scale \(0.1\);
- Adam with learning rate \(10^{-3}\);
- full-batch training;
- 3000 epochs per candidate;
- no early stopping;
- \(\lambda=10^{-7}\);
- seeds \(0,1,2,3,4\).

For each \(\varepsilon\), the notebook reports:
1. correct-selection frequency;
2. target validation loss;
3. validation margin to the best wrong candidate;
4. test loss;
5. mean and sample standard deviation over five seeds.


In [ ]:

import os
import time
import random
import math
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

QUICK_TEST = False

# Repository-relative output directory.
os.makedirs("results/residual_amplitude", exist_ok=True)

SEEDS = [0, 1, 2, 3, 4]
EPS_VALUES = [0.0, 0.05, 0.10, 0.14, 0.155, 0.158]

N_TRAIN = 2000
N_VAL   = 1000
N_TEST  = 1000

EPOCHS = 3000
LR = 1e-3
HIDDEN_WIDTH = 128
RESIDUAL_SCALE = 0.1
# Revised manuscript: primary selector is unpenalized validation loss.
# The original 1e-7 value is retained only in archived/raw results.
LAMBDA_WORD = 0.0
if QUICK_TEST:
    SEEDS = [0]
    EPS_VALUES = [0.0, 0.05, 0.158]
    N_TRAIN = 200
    N_VAL = 100
    N_TEST = 100
    EPOCHS = 20

device = torch.device("cpu")

try:
    torch.use_deterministic_algorithms(True)
except Exception as e:
    print("Could not enforce deterministic algorithms:", e)

EPS_CRITICAL = 1.0 / (2.0 * math.pi)

print("device:", device)
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("epsilon critical = 1/(2*pi) =", EPS_CRITICAL)
print("EPS_VALUES =", EPS_VALUES)
print("QUICK_TEST:", QUICK_TEST)


In [ ]:

# ------------------------------------------------------------
# Diffeomorphism diagnostic for F_epsilon
# ------------------------------------------------------------

diagnostic_rows = []

for eps in EPS_VALUES:
    min_factor = 1.0 - 2.0 * math.pi * eps

    # Since the determinant is a product of the two factors,
    # this is a simple global lower bound in the present regime.
    det_lower_bound = min_factor ** 2

    diagnostic_rows.append({
        "epsilon": eps,
        "1_minus_2pi_epsilon": min_factor,
        "determinant_lower_bound": det_lower_bound,
        "below_critical_epsilon": eps < EPS_CRITICAL,
    })

diagnostic_df = pd.DataFrame(diagnostic_rows)
display(diagnostic_df)

assert diagnostic_df["below_critical_epsilon"].all()


In [ ]:

# ------------------------------------------------------------
# Torus loss
# ------------------------------------------------------------

def torus_diff(pred, target):
    return (pred - target + 0.5) % 1.0 - 0.5

def torus_mse(pred, target):
    return torch.mean(torus_diff(pred, target) ** 2)


In [ ]:

# ------------------------------------------------------------
# Original 9 candidate mapping-class elements
# ------------------------------------------------------------

A = np.array([[1, 1],
              [0, 1]], dtype=np.int64)

A_INV = np.array([[ 1, -1],
                  [ 0,  1]], dtype=np.int64)

B = np.array([[1, 0],
              [1, 1]], dtype=np.int64)

B_INV = np.array([[ 1,  0],
                  [-1,  1]], dtype=np.int64)

TOKEN_MATRIX = {
    "A": A,
    "Ainv": A_INV,
    "B": B,
    "Binv": B_INV,
}

def word_matrix(tokens):
    M = np.eye(2, dtype=np.int64)
    for token in tokens:
        M = M @ TOKEN_MATRIX[token]
    return M

def matrix_map(M):
    Mt = torch.tensor(M.T, dtype=torch.float32, device=device)
    def G(x):
        return (x @ Mt) % 1.0
    return G

candidate_tokens = {
    "id":   [],
    "A":    ["A"],
    "A^-1": ["Ainv"],
    "B":    ["B"],
    "B^-1": ["Binv"],
    "AB":   ["A", "B"],
    "BA":   ["B", "A"],
    "A^2":  ["A", "A"],
    "B^2":  ["B", "B"],
}

candidates = {}

for name, tokens in candidate_tokens.items():
    M = word_matrix(tokens)
    candidates[name] = {
        "tokens": tokens,
        "matrix": M,
        "length": len(tokens),
        "map": matrix_map(M),
    }

# Verify distinct candidate elements.
seen = {}
for name, info in candidates.items():
    key = tuple(info["matrix"].reshape(-1).tolist())
    if key in seen:
        raise ValueError(f"Duplicate element: {seen[key]} and {name}")
    seen[key] = name

candidate_df = pd.DataFrame(
    [
        {
            "candidate": name,
            "length": info["length"],
            "matrix": str(info["matrix"].tolist()),
        }
        for name, info in candidates.items()
    ]
)

display(candidate_df)

HIDDEN_WORD = "AB"


In [ ]:

# ------------------------------------------------------------
# Residual deformation and target
# ------------------------------------------------------------

def F_eps(x, eps):
    u = (
        x[..., 0]
        + eps * torch.sin(2 * torch.pi * (x[..., 0] - x[..., 1]))
    ) % 1.0

    v = (
        x[..., 1]
        + eps * torch.sin(2 * torch.pi * x[..., 1])
    ) % 1.0

    return torch.stack([u, v], dim=-1)

def make_target(eps):
    G_star = candidates[HIDDEN_WORD]["map"]

    def target(x):
        return F_eps(G_star(x), eps=eps)

    return target


In [ ]:

# ------------------------------------------------------------
# Periodic neural network
# ------------------------------------------------------------

class PeriodicResidualNN(torch.nn.Module):
    def __init__(self, hidden=128, residual_scale=0.1):
        super().__init__()
        self.residual_scale = residual_scale

        self.net = torch.nn.Sequential(
            torch.nn.Linear(4, hidden),
            torch.nn.Tanh(),
            torch.nn.Linear(hidden, hidden),
            torch.nn.Tanh(),
            torch.nn.Linear(hidden, 2),
        )

    def periodic_features(self, x):
        return torch.cat(
            [
                torch.cos(2 * torch.pi * x),
                torch.sin(2 * torch.pi * x),
            ],
            dim=-1,
        )

    def forward(self, x):
        features = self.periodic_features(x)
        dx = self.residual_scale * self.net(features)
        return (x + dx) % 1.0


In [ ]:

# ------------------------------------------------------------
# Candidate-wise training
# ------------------------------------------------------------

def train_one_candidate(
    candidate_name,
    x_train,
    y_train,
    x_val,
    y_val,
    x_test,
    y_test,
    seed,
):
    # Same initialization seed across candidates within each repetition.
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    model = PeriodicResidualNN(
        hidden=HIDDEN_WIDTH,
        residual_scale=RESIDUAL_SCALE,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    G = candidates[candidate_name]["map"]

    with torch.no_grad():
        gx_train = G(x_train)
        gx_val = G(x_val)
        gx_test = G(x_test)

    model.train()

    for _ in range(EPOCHS):
        optimizer.zero_grad()

        pred = model(gx_train)
        loss = torus_mse(pred, y_train)

        loss.backward()
        optimizer.step()

    model.eval()

    with torch.no_grad():
        train_loss = torus_mse(model(gx_train), y_train).item()
        val_loss = torus_mse(model(gx_val), y_val).item()
        test_loss = torus_mse(model(gx_test), y_test).item()

    word_length = candidates[candidate_name]["length"]
    penalized_score = val_loss + LAMBDA_WORD * word_length

    return {
        "train_loss": train_loss,
        "val_loss": val_loss,
        "test_loss": test_loss,
        "word_length": word_length,
        "penalized_score": penalized_score,
    }


In [ ]:

# ------------------------------------------------------------
# Run amplitude sweep
# ------------------------------------------------------------

all_rows = []
run_summaries = []

total_trainings = len(SEEDS) * len(EPS_VALUES) * len(candidates)
completed_trainings = 0
global_start = time.time()

for seed in SEEDS:
    # Paired data design:
    # for each seed, the SAME train/val/test inputs are used at every epsilon.
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    x_train = torch.rand(N_TRAIN, 2, device=device)
    x_val   = torch.rand(N_VAL, 2, device=device)
    x_test  = torch.rand(N_TEST, 2, device=device)

    for eps in EPS_VALUES:
        target = make_target(eps)

        with torch.no_grad():
            y_train = target(x_train)
            y_val   = target(x_val)
            y_test  = target(x_test)

        print("=" * 76)
        print(f"seed={seed} | epsilon={eps:.3f} | hidden={HIDDEN_WORD}")
        print("=" * 76)

        rows_this_run = []

        for candidate_name in candidates:
            t0 = time.time()

            result = train_one_candidate(
                candidate_name,
                x_train, y_train,
                x_val, y_val,
                x_test, y_test,
                seed=seed,
            )

            row = {
                "seed": seed,
                "epsilon": eps,
                "candidate": candidate_name,
                **result,
            }

            all_rows.append(row)
            rows_this_run.append(row)

            completed_trainings += 1

            elapsed = time.time() - global_start
            avg_per_training = elapsed / completed_trainings
            remaining = total_trainings - completed_trainings
            eta_min = avg_per_training * remaining / 60.0

            print(
                f"{candidate_name:6s} "
                f"val={result['val_loss']:.4e} "
                f"score={result['penalized_score']:.4e} "
                f"({time.time()-t0:.1f}s; ETA ~{eta_min:.1f} min)"
            )

        run_df = pd.DataFrame(rows_this_run)

        target_row = run_df[
            run_df["candidate"] == HIDDEN_WORD
        ].iloc[0]

        wrong_df = run_df[
            run_df["candidate"] != HIDDEN_WORD
        ]

        best_wrong_idx = wrong_df["val_loss"].idxmin()
        best_wrong_row = wrong_df.loc[best_wrong_idx]

        validation_margin = (
            best_wrong_row["val_loss"]
            - target_row["val_loss"]
        )

        selected_penalized = run_df.loc[
            run_df["penalized_score"].idxmin(),
            "candidate"
        ]

        selected_unpenalized = run_df.loc[
            run_df["val_loss"].idxmin(),
            "candidate"
        ]

        selected_test_loss = run_df.loc[
            run_df["penalized_score"].idxmin(),
            "test_loss"
        ]

        run_summary = {
            "seed": seed,
            "epsilon": eps,
            "selected_penalized": selected_penalized,
            "selected_unpenalized": selected_unpenalized,
            "correct_penalized": selected_penalized == HIDDEN_WORD,
            "correct_unpenalized": selected_unpenalized == HIDDEN_WORD,
            "target_val_loss": target_row["val_loss"],
            "best_wrong_candidate": best_wrong_row["candidate"],
            "best_wrong_val_loss": best_wrong_row["val_loss"],
            "validation_margin": validation_margin,
            "selected_test_loss": selected_test_loss,
        }

        run_summaries.append(run_summary)

        print("\nRun summary:")
        print(run_summary)

        # Autosave after every epsilon/seed combination.
        pd.DataFrame(all_rows).to_csv(
            "results/residual_amplitude/residual_amplitude_all_candidate_results.csv",
            index=False,
        )

        pd.DataFrame(run_summaries).to_csv(
            "results/residual_amplitude/residual_amplitude_run_summaries.csv",
            index=False,
        )

print("\nFull amplitude sweep completed.")


In [ ]:

# ------------------------------------------------------------
# Per-run results
# ------------------------------------------------------------

run_summary_df = pd.DataFrame(run_summaries)

display(
    run_summary_df.sort_values(
        ["epsilon", "seed"]
    )
)

print("\nSelection counts (lambda = 0):")
display(
    pd.crosstab(
        run_summary_df["epsilon"],
        run_summary_df["selected_penalized"],
    )
)


In [ ]:

# ------------------------------------------------------------
# Aggregate across seeds
# ------------------------------------------------------------

def sample_sd(x):
    x = np.asarray(x, dtype=float)
    return np.std(x, ddof=1) if len(x) > 1 else np.nan

summary_rows = []

for eps in EPS_VALUES:
    d = run_summary_df[
        run_summary_df["epsilon"] == eps
    ]

    min_factor = 1.0 - 2.0 * math.pi * eps
    det_lower_bound = min_factor ** 2

    summary_rows.append({
        "epsilon": eps,
        "1_minus_2pi_epsilon": min_factor,
        "determinant_lower_bound": det_lower_bound,
        "correct_penalized_count": int(d["correct_penalized"].sum()),
        "n_seeds": len(d),
        "selection_rate": float(d["correct_penalized"].mean()),
        "target_val_mean": float(d["target_val_loss"].mean()),
        "target_val_sd": sample_sd(d["target_val_loss"]),
        "margin_mean": float(d["validation_margin"].mean()),
        "margin_sd": sample_sd(d["validation_margin"]),
        "test_loss_mean": float(d["selected_test_loss"].mean()),
        "test_loss_sd": sample_sd(d["selected_test_loss"]),
    })

summary_df = pd.DataFrame(summary_rows)

display(summary_df)

summary_df.to_csv(
    "results/residual_amplitude/residual_amplitude_aggregate_summary.csv",
    index=False,
)


In [ ]:

# ------------------------------------------------------------
# Manuscript-friendly table
# ------------------------------------------------------------

paper_rows = []

for _, r in summary_df.iterrows():
    paper_rows.append({
        "epsilon": f"{r['epsilon']:.3f}",
        "1-2pi epsilon": f"{r['1_minus_2pi_epsilon']:.4f}",
        "Selected correctly": f"{int(r['correct_penalized_count'])}/{int(r['n_seeds'])}",
        "Val. loss (mean ± SD)": (
            f"{r['target_val_mean']:.3e} ± "
            f"{r['target_val_sd']:.3e}"
        ),
        "Margin (mean ± SD)": (
            f"{r['margin_mean']:.3e} ± "
            f"{r['margin_sd']:.3e}"
        ),
        "Test loss (mean ± SD)": (
            f"{r['test_loss_mean']:.3e} ± "
            f"{r['test_loss_sd']:.3e}"
        ),
    })

paper_table_df = pd.DataFrame(paper_rows)

display(paper_table_df)

paper_table_df.to_csv(
    "results/residual_amplitude/residual_amplitude_paper_table.csv",
    index=False,
)


In [ ]:

# ------------------------------------------------------------
# Plot 1: validation margin versus residual amplitude
# ------------------------------------------------------------

plt.figure(figsize=(7, 5))
plt.errorbar(
    summary_df["epsilon"],
    summary_df["margin_mean"],
    yerr=summary_df["margin_sd"],
    marker="o",
    capsize=4,
)
plt.axhline(0, linewidth=1)
plt.axvline(EPS_CRITICAL, linestyle="--", linewidth=1)
plt.xlabel("Residual amplitude epsilon")
plt.ylabel("Validation margin")
plt.title("Validation margin versus residual amplitude")
plt.grid(True)
plt.show()


In [ ]:

# ------------------------------------------------------------
# Plot 2: target validation loss versus residual amplitude
# ------------------------------------------------------------

plt.figure(figsize=(7, 5))
plt.errorbar(
    summary_df["epsilon"],
    summary_df["target_val_mean"],
    yerr=summary_df["target_val_sd"],
    marker="o",
    capsize=4,
)
plt.axvline(EPS_CRITICAL, linestyle="--", linewidth=1)
plt.yscale("log")
plt.xlabel("Residual amplitude epsilon")
plt.ylabel("Target-candidate validation loss")
plt.title("Target validation loss versus residual amplitude")
plt.grid(True)
plt.show()



## Interpretation guide

The key distinction is between **fit quality** and **mapping-class selection**.

As \(\varepsilon\) approaches \(1/(2\pi)\), the residual deformation becomes increasingly strong and \(F_\varepsilon\) approaches a near-singular diffeomorphism. It is therefore reasonable for the target-candidate PNN validation loss to increase.

The more important selection quantity is
\[
\widehat\gamma
=
\min_{G\neq AB}L_{\rm val}(G)
-
L_{\rm val}(AB).
\]

- If \(\widehat\gamma>0\), the target mapping-class candidate still has the best unpenalized validation loss.
- If all five seeds remain correct up to \(\varepsilon=0.158\), the appropriate conclusion is **not** that the method never breaks. Rather, the experiment shows stable selection throughout the tested diffeomorphic regime, including amplitudes very close to the boundary \(\varepsilon=1/(2\pi)\).
- If selection fails for one or more seeds near the boundary, that provides a natural empirical breakdown point to report.
